In [1]:
!pip install tensorflow


Defaulting to user installation because normal site-packages is not writeable


In [2]:
import sys
print(sys.executable)


C:\ProgramData\anaconda3\python.exe


In [3]:
import tensorflow as tf

In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
import os
import shutil
from PIL import Image
import numpy as np

In [5]:
print("hello")

hello


In [6]:
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8,1.2],
    fill_mode='nearest'
)

In [7]:
folder_path = "C:/projects/Fake-Currency-Checker/Indian Currency Dataset"
train = os.path.join(folder_path, 'train')
test = os.path.join(folder_path, 'test')
print("hello")

hello


In [8]:
train_fake = os.path.join(train, 'fake')
train_real = os.path.join(train, 'real')

In [9]:
print(os.path.exists("C:/projects/Fake-Currency-Checker/Indian Currency Dataset/train/fake"))

True


In [10]:
train_fake_images = [os.path.join(train_fake, img) for img in os.listdir(train_fake)]
train_real_images = [os.path.join(train_real, img) for img in os.listdir(train_real)]

In [11]:
train_fake, val_fake = train_test_split(train_fake_images, test_size=0.15, random_state=42)
train_real, val_real = train_test_split(train_real_images, test_size=0.15, random_state=42)

In [12]:
train_images = train_fake + train_real
val_images = val_fake + val_real

In [13]:
validation = os.path.join(folder_path, 'validation')
os.makedirs(validation, exist_ok=True)

In [14]:
import os
import shutil
from sklearn.model_selection import train_test_split

base_path = r"C:\projects\Fake-Currency-Checker\Indian Currency Dataset"

train_dir = os.path.join(base_path, "train")
validation_dir = os.path.join(base_path, "validation")

train_fake_dir = os.path.join(train_dir, "fake")
train_real_dir = os.path.join(train_dir, "real")

val_fake_dir = os.path.join(validation_dir, "fake")
val_real_dir = os.path.join(validation_dir, "real")

# create validation folders
os.makedirs(val_fake_dir, exist_ok=True)
os.makedirs(val_real_dir, exist_ok=True)

# get all images
train_fake_images = [os.path.join(train_fake_dir, f) for f in os.listdir(train_fake_dir)]
train_real_images = [os.path.join(train_real_dir, f) for f in os.listdir(train_real_dir)]

# split data
train_fake_split, val_fake_split = train_test_split(train_fake_images, test_size=0.15, random_state=42)
train_real_split, val_real_split = train_test_split(train_real_images, test_size=0.15, random_state=42)

# move validation fake images
for img_path in val_fake_split:
    shutil.move(img_path, os.path.join(val_fake_dir, os.path.basename(img_path)))

# move validation real images
for img_path in val_real_split:
    shutil.move(img_path, os.path.join(val_real_dir, os.path.basename(img_path)))


In [15]:
print(f'Train set size: {len(train_images)}')
print(f'Validation set size: {len(val_images)}')

Train set size: 927
Validation set size: 165


In [16]:
batch_size = 32
target_size = (224, 224)
train_generator = datagen.flow_from_directory(
    train,
    target_size=target_size,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=True,
    seed=42
)

validation_generator = datagen.flow_from_directory(
    validation,
    target_size=target_size,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False,
    seed=42
)

test_generator = datagen.flow_from_directory(
    test,
    target_size=target_size,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False,
    seed=42
)

Found 927 images belonging to 2 classes.
Found 2037 images belonging to 2 classes.
Found 107 images belonging to 2 classes.


In [17]:
print(os.listdir(train))
print(os.listdir(validation))
print(os.listdir(test))
print("hello")

['fake', 'real']
['fake', 'real']
['fake', 'real']
hello


In [18]:
model = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False)
model.trainable = False

In [19]:
flatten_layer = tf.keras.layers.Flatten()(model.output)
dropout_layer = tf.keras.layers.Dropout(0.5)(flatten_layer)
output_layer = tf.keras.layers.Dense(1, activation='sigmoid')(dropout_layer)

model = tf.keras.models.Model(model.input, output_layer)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [20]:
num_epochs = 5
model.fit(train_generator, epochs=num_epochs, validation_data=validation_generator)

Epoch 1/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 99s 3s/step - accuracy: 0.7120 - loss: 1.2638 - val_accuracy: 0.7683 - val_loss: 1.0205
Epoch 2/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 82s 3s/step - accuracy: 0.8220 - loss: 0.8398 - val_accuracy: 0.8198 - val_loss: 0.7937
Epoch 3/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 81s 3s/step - accuracy: 0.7918 - loss: 1.0974 - val_accuracy: 0.7815 - val_loss: 1.1069
Epoch 4/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 79s 3s/step - accuracy: 0.8058 - loss: 0.8875 - val_accuracy: 0.8228 - val_loss: 0.7700
Epoch 5/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 106s 4s/step - accuracy: 0.8198 - loss: 0.8625 - val_accuracy: 0.8326 - val_loss: 0.9174


In [21]:
test_loss, test_accuracy = model.evaluate(test_generator)
print(f'Test Loss: {test_loss}, Test Accuracy: {test_accuracy}')

4/4 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.7009 - loss: 1.6038
Test Loss: 1.6037639379501343, Test Accuracy: 0.7009345889091492


In [22]:
train_loss, train_accuracy = model.evaluate(train_generator)
print(f'Train Loss: {train_loss}, Train Accuracy: {train_accuracy}')

29/29 ━━━━━━━━━━━━━━━━━━━━ 43s 2s/step - accuracy: 0.8706 - loss: 0.7416
Train Loss: 0.7416329383850098, Train Accuracy: 0.8705501556396484


In [23]:
val_loss, val_accuracy = model.evaluate(validation_generator)
print(f'Validation Loss: {val_loss}, Validation Accuracy: {val_accuracy}')

64/64 ━━━━━━━━━━━━━━━━━━━━ 100s 2s/step - accuracy: 0.8429 - loss: 0.8321
Validation Loss: 0.8320891857147217, Validation Accuracy: 0.8429062366485596


In [24]:
predictions = model.predict(test_generator)

4/4 ━━━━━━━━━━━━━━━━━━━━ 14s 2s/step


In [25]:
def load_and_preprocess_image(image_path, target_size=(224, 224)):
    img = Image.open(image_path)
    img = img.resize(target_size)
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    return img_array

In [26]:
def predict_currency_gradio(image):
    img = Image.fromarray(image)
    img = img.resize((224,224))
    
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)

    if prediction[0][0] >= 0.5:
        return "✅ Real Currency"
    else:
        return "❌ Fake Currency"

In [30]:
y_pred = model.predict(validation_generator)   # or test_generator
y_pred = (y_pred > 0.5).astype(int)

64/64 ━━━━━━━━━━━━━━━━━━━━ 100s 2s/step


In [31]:
y_true = validation_generator.classes   # or test_generator.classes

In [ ]:
image_path = "C:/Users/mbhav/Downloads/download.jpg"

result = predict_currency(image_path, model)
print(f"The predicted result is: {result}")

In [32]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [33]:
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

In [34]:
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.8414334806087383
Precision: 0.8610526315789474
Recall: 0.8107036669970268
F1 Score: 0.8351199591628382


In [ ]:
!pip install gradio

In [27]:
import gradio as gr

In [28]:
interface = gr.Interface(
    fn=predict_currency_gradio,
    inputs=gr.Image(type="numpy"),
    outputs="text",
    title="Fake Currency Detection System",
    description="Upload an image of Indian currency to check if it is Real or Fake"
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
